# BRD knowledge-gap prototype
Synthetic regulation + BRD. Cells 1-9b need no internet. Cells 10-17 call a hosted LLM and need a Groq key stored as a secret. Run top to bottom once; afterwards a single cell can be re-run on its own (state lives in the kernel). Restarted the kernel? Re-run from Cell 1.
All data here is synthetic and prompts were tuned on it, so scores show the method works end to end, not its accuracy on real documents.
LLM output varies between runs (the clarification check flagged 12 requirements in one run and all 14 in another), so compare runs before trusting a single number.

## Cell 1 - setup + synthetic regulation

In [ ]:
import re, math, copy
from collections import Counter

REG = {
 "R1":"Submit a Position Report daily by 18:00 UTC for each open position in a Reportable Instrument.",
 "R2":"Reportable Instrument means a derivative contract listed in Annex I.",
 "R3":"Report shall include counterparty LEI, notional amount, valuation, trade date.",
 "R4":"Valuation shall be mark-to-market at end-of-day close.",
 "R5":"Notional expressed in EUR, converted at the ECB reference rate.",
 "R6":"Corrections shall be resubmitted within 1 business day of error discovery.",
 "R7":"Reports shall be submitted in XML per the schema published by the regulator.",
}


## Cell 2 - synthetic BRD (stubbed extraction) + ground truth

In [ ]:
def E(name,kind,**slots): return dict(name=name,kind=kind,slots=slots)  # slots: key=(value, evidence_quote)
BRD = {
 "B1":dict(text="Generate daily Position Report by 17:30 UTC (30-min buffer for gateway retries).",derived_from=["R1"],rationale="30-min buffer for gateway retries",requires=[]),
 "B2":dict(text="Include only positions in products classified as Reportable Product.",derived_from=["R2"],rationale=None,requires=[E("Product->Annex I mapping","mapping")]),
 "B3":dict(text="Populate counterparty LEI from Counterparty Master.",derived_from=["R3"],rationale=None,requires=[E("Counterparty LEI","data_element",source_system=("Counterparty Master","Counterparty Master"))]),
 "B4":dict(text="Calculate notional in EUR using ECB reference rate.",derived_from=["R5"],rationale=None,requires=[E("Notional EUR","calculation",formula=("notional*rate","notional in EUR using ECB reference rate")),E("ECB reference rate","data_element")]),
 "B5":dict(text="Populate valuation using end-of-day mark-to-market price.",derived_from=["R4"],rationale=None,requires=[E("EOD market price","data_element")]),
 "B6":dict(text="Exclude intra-group trades from reporting.",derived_from=[],rationale=None,requires=[]),
 "B7":dict(text="Support correction resubmission within 1 business day.",derived_from=["R6"],rationale=None,requires=[E("Business day calendar","reference_data")]),
 "B8":dict(text="Submit via Regulator Gateway using XML schema v2.1.",derived_from=["R7"],rationale=None,requires=[E("Regulator Gateway XML schema v2.1","system_capability")]),
 "B9":dict(text="Map legacy TradeVault trades via Legacy Mapping Table.",derived_from=[],rationale="legacy platform onboarding",requires=[E("Legacy Mapping Table","mapping")]),
 "B10":dict(text="Report only positions above the materiality threshold.",derived_from=["R1"],rationale=None,requires=[]),
 "B11":dict(text="Round values using standard house rounding as per current practice.",derived_from=["R5"],rationale=None,requires=[]),
 "B12":dict(text="Populate trade date from the Trade Booking System.",derived_from=["R3"],rationale=None,requires=[E("Trade date","data_element",source_system=("Trade Booking System","Trade Booking System"))]),
 "B13":dict(text="Aggregate positions by counterparty before reporting.",derived_from=["R3"],rationale=None,requires=[]),   # hidden: aggregation/netting rule undefined
 "B14":dict(text="Populate trade date in ISO format.",derived_from=["R3"],rationale=None,requires=[]),                     # clean, but uses new words
}
TRUTH_GAP_REQS={"B2","B4","B5","B6","B7","B8","B9","B10","B11","B13"}
CLEAN_REQS=set(BRD)-TRUTH_GAP_REQS
REQ_SLOTS={"data_element":["source_system"],"calculation":["formula","rounding"],"mapping":["definition"],"reference_data":["source_system"],"system_capability":["interface_spec"]}


## Cell 3 - Method 1: slot check with evidence validation

In [ ]:
# ---- 1. slot check with evidence-span validation ----
def m1_slots(brd=BRD):
    out=[]
    for r,d in brd.items():
        for e in d["requires"]:
            for s in REQ_SLOTS[e["kind"]]:
                v=e["slots"].get(s)
                if v is None: out.append((r,e["name"],e["kind"],"missing "+s))
                elif v[1] not in d["text"]: out.append((r,e["name"],e["kind"],f"unverified {s} (evidence not in BRD text)"))
    return out


## Cell 4 - Method 2: provenance + deviation detection

In [ ]:
# ---- 2. provenance + automatic deviation/added-parameter detection ----
def params(t):
    return {"time":set(re.findall(r"\b\d{1,2}:\d{2}\b",t)),"duration":set(re.findall(r"\b\d+\s+(?:business\s+)?days?\b",t.lower())),
            "currency":set(re.findall(r"\b(?:EUR|USD|GBP)\b",t)),"version":set(re.findall(r"\bv\d+(?:\.\d+)*\b",t))}
def deviations(r,d):
    c={k:set() for k in ("time","duration","currency","version")}
    for cl in d["derived_from"]:
        for k,v in params(REG[cl]).items(): c[k]|=v
    return [k for k,b in params(d["text"]).items() if b and not b<=c[k]]
def m2_prov(brd=BRD):
    out=[]
    for r,d in brd.items():
        if not d["derived_from"] and not d["rationale"]: out.append((r,"rationale:"+r,"interpretation","no clause and no rationale"))
        dv=deviations(r,d)
        if dv and not d["rationale"]: out.append((r,"rationale:"+r,"interpretation",f"undocumented deviation/added parameter: {dv}"))
    return out


## Cell 5 - Method 3: implicit-knowledge proxy

In [ ]:
# ---- 3. implicit-knowledge proxy ----
STOP=set("a an the of in on at by for to from with within via as per and or is are be shall above before using only include populate generate support use apply system".split())
VAGUE=re.compile(r"as per current practice|\bstandard\b|\bexisting\b|\busual\b|as before|as agreed|business as usual",re.I)
stem=lambda w:w[:5]
words=lambda t:[w for w in re.findall(r"[a-z]+",t.lower()) if w not in STOP and len(w)>2]
def m3_lex(brd=BRD):
    out=[]
    for r,d in brd.items():
        cov=set()
        for c in d["derived_from"]: cov|={stem(w) for w in words(REG[c])}
        for e in d["requires"]: cov|={stem(w) for w in words(e["name"])}
        if d["rationale"]: cov|={stem(w) for w in words(d["rationale"])}
        unc=sorted({w for w in words(d["text"]) if stem(w) not in cov})
        if unc or VAGUE.search(d["text"]): out.append((r,"implicit:"+r,"implicit",f"uncovered terms {unc}"+("; vague phrase" if VAGUE.search(d["text"]) else "")))
    return out


## Cell 6 - compare methods

In [ ]:
def score(name,flags):
    f={x[0] for x in flags}; tp=len(f&TRUTH_GAP_REQS); fp=sorted(f&CLEAN_REQS); fn=sorted(TRUTH_GAP_REQS-f)
    p=tp/len(f) if f else 0; rc=tp/len(TRUTH_GAP_REQS)
    print(f"{name:26} P={p:.2f} R={rc:.2f}  FP={fp}  missed={fn}")
    return f
print("== METHOD COMPARISON (requirement level, 14 reqs, 10 truly gapped) ==")
a=m1_slots(); b=m2_prov(); c=m3_lex()
fa=score("M1 slots+evidence",a); fb=score("M2 provenance+deviation",b); fc=score("M3 lexical implicit",c)
allf=score("HYBRID (union)",a+b+c)
print("\nB1 (documented deviation) flagged by M2?", any(x[0]=="B1" for x in b), "| deviation auto-detected:", deviations("B1",BRD["B1"]))
print("Only M3 catches:",sorted(fc-fa-fb)," | Missed by ALL:",sorted(TRUTH_GAP_REQS-allf))


## Cell 7 - robustness tests (hallucinated vs grounded slot)

In [ ]:
# ---- robustness test: hallucinated slot must not close a gap ----
t=copy.deepcopy(BRD); t["B2"]["requires"][0]["slots"]["definition"]=("mapping sheet v3","see mapping sheet v3")
print("\nHallucinated slot test:",[x[3] for x in m1_slots(t) if x[0]=="B2"])
t=copy.deepcopy(BRD); t["B2"]["requires"][0]["slots"]["definition"]=("classification table","products classified as Reportable Product")
print("Grounded slot test (gap should close):",[x for x in m1_slots(t) if x[0]=="B2"])


## Cell 8 - synthetic internal document corpus + routing prior

In [ ]:
# ---- 5. routing: keyword-only vs typed prior ----
DOCS={"D1":("catalog","Product catalog: product taxonomy, classification, instrument types, Annex I eligibility list"),
 "D2":("dictionary","Data dictionary: counterparty master, LEI, field definitions, source systems, data owners"),
 "D3":("platform_spec","Market data platform spec: ECB FX reference rate feed, end-of-day close prices, pricing snapshots"),
 "D4":("spreadsheet","Existing reporting spreadsheet: notional conversion, rounding, decimal places, FX"),
 "D5":("memo","Compliance memo: intra-group trades exemption interpretation, legal opinion, regulator correspondence"),
 "D6":("platform_spec","Business day calendar service: TARGET2 calendar, working days, reference data"),
 "D7":("integration_spec","Gateway integration spec: regulator gateway API, XML schema versions, message formats"),
 "D8":("migration","Legacy TradeVault migration notes: legacy mapping table, field mapping, onboarding"),
 "D9":("hr","HR onboarding handbook: leave policy, expenses, joining formalities"),
 "D10":("minutes","Meeting minutes: quarterly budget planning and vendor contracts"),
 "D11":("policy","Reporting policy: materiality threshold for position reporting, reportable position size"),
 "D12":("style_guide","Reporting conventions: house rounding rules, decimal places, number formats"),
 "D13":("policy","Audit policy: materiality threshold for audit sampling and financial statements")}
PRIOR={"mapping":["catalog","dictionary","migration"],"data_element":["dictionary","platform_spec"],"calculation":["spreadsheet","style_guide","policy"],
 "reference_data":["platform_spec","dictionary"],"system_capability":["integration_spec"],"interpretation":["policy","memo"],"implicit":["policy","style_guide","dictionary"]}
TRUTH_SRC={"Product->Annex I mapping":"D1","Notional EUR":"D4","ECB reference rate":"D3","EOD market price":"D3","rationale:B6":"D5",
 "Business day calendar":"D6","Regulator Gateway XML schema v2.1":"D7","Legacy Mapping Table":"D8","implicit:B10":"D11","implicit:B11":"D12"}
tok=lambda s:re.findall(r"[a-z0-9]+",s.lower()); N=len(DOCS)
df=Counter(w for _,t in DOCS.values() for w in set(tok(t)))
vec=lambda s:{w:c*math.log((N+1)/(df.get(w,0)+.5)) for w,c in Counter(tok(s)).items()}
cos=lambda x,y:sum(x[w]*y.get(w,0) for w in x)/(math.sqrt(sum(v*v for v in x.values()))*math.sqrt(sum(v*v for v in y.values()))+1e-9)


## Cell 9 - routing: keyword-only vs typed prior

In [ ]:
gaps={}
for g in a+b+c: gaps.setdefault(g[1],g)
print("\n== ROUTING ==")
for mode in ("keyword-only","typed-prior"):
    h1=h3=0
    for ent,truth in TRUTH_SRC.items():
        r,_,kind,why=gaps[ent]; q=vec(f"{BRD[r]['text']} {ent} {why}")
        sc={d:cos(q,vec(t))+(0.15 if mode=="typed-prior" and ty in PRIOR[kind] else 0) for d,(ty,t) in DOCS.items()}
        rk=sorted(sc,key=lambda d:-sc[d])[:3]; h1+=rk[0]==truth; h3+=truth in rk
    print(f"{mode:13} top-1={h1}/{len(TRUTH_SRC)} top-3={h3}/{len(TRUTH_SRC)}")


## Cell 9b - build and draw the actual knowledge graph (saves graph.json + graph.png)
Run after cells 1-9. Needs no internet.

In [ ]:
import networkx as nx, json
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

G = nx.DiGraph()
for cid, t in REG.items(): G.add_node(cid, type="clause", label=cid, text=t)
for rid, d in BRD.items():
    G.add_node(rid, type="requirement", label=rid, text=d["text"])
    for cl in d["derived_from"]: G.add_edge(rid, cl, rel="derived_from")
    for e in d["requires"]:
        G.add_node(e["name"], type=e["kind"], label=e["name"])
        G.add_edge(rid, e["name"], rel="requires")
# mark gaps (high-confidence = M1+M2, review = M3) on the nodes they attach to
hi = {x[1] for x in a+b}; rev = {x[1] for x in c}
for n in G.nodes:
    G.nodes[n]["gap"] = "high" if (n in hi) or (f"rationale:{n}" in hi) else ("review" if f"implicit:{n}" in rev else "none")
# label provenance on each requirement
for rid, d in BRD.items():
    G.nodes[rid]["provenance"] = "business-added" if not d["derived_from"] else ("interpreted" if deviations(rid, d) else "regulatory")
json.dump(nx.node_link_data(G, edges="links"), open("graph.json","w"), indent=1)

color = {"clause":"#9ecae1","requirement":"#d9d9d9"}
def col(n):
    g = G.nodes[n]["gap"]
    if g == "high": return "#e6550d"
    if g == "review": return "#fdae6b"
    return color.get(G.nodes[n]["type"], "#c7e9c0")
pos = nx.spring_layout(G, seed=3, k=0.9)
plt.figure(figsize=(13,9))
nx.draw_networkx_edges(G, pos, arrows=True, alpha=.4, arrowsize=10)
nx.draw_networkx_nodes(G, pos, node_color=[col(n) for n in G.nodes], node_size=900)
nx.draw_networkx_labels(G, pos, {n:(n if len(n)<8 else n[:18]) for n in G.nodes}, font_size=7)
plt.title("Orange = high-confidence gap, light orange = review, blue = regulation clause, grey = BRD requirement, green = dependency")
plt.axis("off"); plt.savefig("graph.png", dpi=130, bbox_inches="tight"); plt.show()
print(G.number_of_nodes(), "nodes,", G.number_of_edges(), "edges |",
      Counter(G.nodes[r]["provenance"] for r in BRD))


## Cell 10 - LLM client (needs Internet ON and a secret named GROQ_API_KEY; never paste a key into the notebook)

In [ ]:
!pip -q install groq
import os, json, time
from groq import Groq
try:
    from kaggle_secrets import UserSecretsClient
    GROQ_KEY = UserSecretsClient().get_secret("GROQ_API_KEY")   # Kaggle: Add-ons > Secrets
except Exception:
    GROQ_KEY = os.environ.get("GROQ_API_KEY")
client = Groq(api_key=GROQ_KEY)

available = sorted(m.id for m in client.models.list().data)
print(available)
PREFERRED = ["openai/gpt-oss-120b", "llama-3.3-70b-versatile", "qwen/qwen3-32b", "openai/gpt-oss-20b", "llama-3.1-8b-instant"]
MODEL = next((m for m in PREFERRED if m in available), None)
assert MODEL, f"None of the preferred models are available. Choose one from: {available}"
print("Using:", MODEL)

def ask_json(system, user, tries=5):
    kwargs = dict(model=MODEL, temperature=0,
                  messages=[{"role":"system","content":system},{"role":"user","content":user}])
    for i in range(tries):
        try:
            try:
                r = client.chat.completions.create(response_format={"type":"json_object"}, **kwargs)
            except Exception as e:
                if "429" in str(e): raise
                r = client.chat.completions.create(**kwargs)
            txt = r.choices[0].message.content
            return json.loads(txt[txt.index("{"): txt.rindex("}")+1])
        except Exception as e:
            if "429" in str(e):
                wait = 15*(i+1); print("rate limit, waiting", wait, "s"); time.sleep(wait); continue
            raise
    raise RuntimeError("gave up after repeated rate limits")


## Cell 11 - input text for the LLM steps (defaults to the synthetic example; replace to test your own)

In [ ]:
# Input for the LLM steps. By default: the synthetic regulation and BRD built in cells 1-2.
# To test your own text, replace these two variables (keep ids stable; "clauses" = the clauses a requirement derives from).
REAL_REG = REG
REAL_BRD = {k: {"text": v["text"], "clauses": v["derived_from"]} for k, v in BRD.items()}


## Cell 12 - LLM extraction with quote validation

In [ ]:
EXTRACT_SYS = """You extract structured knowledge from ONE BRD requirement. Return JSON:
{"rationale": str|null,
 "requires": [{"name": str, "kind": "data_element|calculation|mapping|reference_data|system_capability",
               "slots": {"<slot>": {"value": str, "evidence_quote": str}}}]}
Slots: data_element/reference_data -> source_system (an INTERNAL system, table or team that supplies it; an external publisher is NOT a source system); calculation -> formula, rounding; mapping -> definition (the actual mapping rules or contents); system_capability -> interface_spec (an actual specification, not just a system name).
Also list every data input the requirement uses (e.g. prices, rates, calendars), even if the source is not stated.
Every slot MUST have evidence_quote copied VERBATIM from the requirement text. If the text only NAMES something without defining it, OMIT the slot. Never guess."""

REAL_EXTRACTED = {}
for qid, q in REAL_BRD.items():
    out = ask_json(EXTRACT_SYS, f"Requirement: {q['text']}")
    for e in out.get("requires", []):          # quote validation: keep only slots whose quote is literally in the text
        e["slots"] = {k:(v["value"], v["evidence_quote"]) for k,v in e.get("slots",{}).items()
                      if v.get("evidence_quote","") in q["text"]}
    REAL_EXTRACTED[qid] = out
    print(qid, json.dumps(out)[:300])
    time.sleep(1)


## Cell 13 - extraction-based gap check, scored against the baseline

In [ ]:
SPEC_WORDS = ("spec","api","endpoint","document","wsdl","xsd","interface")
def weak(slot, quote, text):
    if len(quote) >= 0.8*len(text): return True              # quote is basically the whole requirement
    if VAGUE.search(quote): return True                      # "standard", "current practice"...
    if slot == "interface_spec" and not any(w in quote.lower() for w in SPEC_WORDS): return True
    return False
LLM_BRD = {}
for q, d in REAL_BRD.items():
    reqs = []
    for e in REAL_EXTRACTED[q].get("requires", []):
        if e["kind"] not in REQ_SLOTS: continue
        e = dict(e); e["slots"] = {k:v for k,v in e["slots"].items() if not weak(k, v[1], d["text"])}
        reqs.append(e)
    LLM_BRD[q] = {"text": d["text"], "derived_from": d["clauses"],
                  "rationale": REAL_EXTRACTED[q].get("rationale"), "requires": reqs}
slot_flag = {x[0] for x in m1_slots(LLM_BRD)}
tp = len(slot_flag & TRUTH_GAP_REQS)
print("Extraction-based check  P=%.2f  R=%.2f" % (tp/len(slot_flag) if slot_flag else 0, tp/len(TRUTH_GAP_REQS)))
print("false positives:", sorted(slot_flag & CLEAN_REQS), "| missed:", sorted(TRUTH_GAP_REQS - slot_flag))
print("flag-everything baseline: P=%.2f R=1.00" % (len(TRUTH_GAP_REQS)/len(BRD)))


## Cell 14 - LLM derivability list (over-flags; shown for comparison)

In [ ]:
ALLOWED = {"definition","data_source","business_rule","calculation","interpretation"}
norm = lambda s: s.lower().replace("\u2011","-").replace("\u202f"," ").strip()
DERIV_SYS = """You are a regulatory business analyst. Given ONLY the linked regulation clauses and the requirement text,
list the most critical knowledge that is missing to implement the requirement. Return at most 3 items.
Use only these types: definition, data_source, business_rule, calculation, interpretation.
Do NOT list generic items (ability of the system to run, format mappings, error handling). Ignore common standards (ISO, XML)
and anything the requirement itself defines. trigger_words must be copied verbatim from the requirement or clauses.
If nothing critical is missing return {"derivable": true, "missing": []}.
Format: {"derivable": bool, "missing": [{"type": str, "what": str, "trigger_words": str}]}"""
DERIV = {}
for qid, q in REAL_BRD.items():
    clauses = "\n".join(f"{c}: {REAL_REG[c]}" for c in (q["clauses"] or list(REAL_REG)))
    hay = norm(q["text"] + " " + clauses)
    res = ask_json(DERIV_SYS, f"Clauses:\n{clauses}\n\nRequirement: {q['text']}")
    res["missing"] = [m for m in res.get("missing", []) if m.get("type") in ALLOWED and m.get("trigger_words","").strip() and
        all(norm(p) in hay for p in m.get("trigger_words","").split(",") if p.strip())]
    DERIV[qid] = res
    print(qid, [(m["type"], m["what"][:50]) for m in res["missing"]])
    time.sleep(2)
flag = {q for q, r in DERIV.items() if r.get("missing")}
tp = len(flag & TRUTH_GAP_REQS)
print("LLM derivability  P=%.2f  R=%.2f" % (tp/len(flag) if flag else 0, tp/len(TRUTH_GAP_REQS)))
print("false positives:", sorted(flag & CLEAN_REQS), "| missed:", sorted(TRUTH_GAP_REQS - flag))


## Cell 15 - LLM 'needs clarification?' check

In [ ]:
BIN_SYS = """You review ONE BRD requirement against its linked regulation clauses.
Question: could a competent developer implement this requirement exactly as written without asking anyone for a missing definition, data source, rule or value?
Standard formats (ISO dates, XML) and anything the requirement itself states are NOT missing. Be strict but not pedantic.
Example (clean): "Store the filing date in ISO 8601 format in column FILING_DATE of table T_FILING."
Example (unclear): "Flag high-value filings."
Return JSON: {"needs_clarification": true or false, "question": "the single most important question, or empty"}"""
BIN = {}
for qid, q in REAL_BRD.items():
    clauses = "\n".join(f"{c}: {REAL_REG[c]}" for c in q["clauses"]) or "(none linked)"
    BIN[qid] = ask_json(BIN_SYS, f"Clauses:\n{clauses}\n\nRequirement: {q['text']}")
    print(qid, BIN[qid]); time.sleep(2)
bin_flag = {q for q, r in BIN.items() if r.get("needs_clarification")}
tp = len(bin_flag & TRUTH_GAP_REQS)
print("Clarification check  P=%.2f  R=%.2f" % (tp/len(bin_flag) if bin_flag else 0, tp/len(TRUTH_GAP_REQS)))
print("false positives:", sorted(bin_flag & CLEAN_REQS), "| missed:", sorted(TRUTH_GAP_REQS - bin_flag))


## Cell 16 - combine the two checks (agreement = high confidence)

In [ ]:
for name, f in [("both agree", slot_flag & bin_flag), ("either flags", slot_flag | bin_flag)]:
    tp = len(f & TRUTH_GAP_REQS)
    print(f"{name:13} flagged={len(f)}  P={(tp/len(f) if f else 0):.2f}  R={tp/len(TRUTH_GAP_REQS):.2f}  FP={sorted(f & CLEAN_REQS)}")
print("only extraction check:", sorted(slot_flag - bin_flag), "| only clarification check:", sorted(bin_flag - slot_flag))


## Cell 17 - export gap_report.csv / .xlsx

In [ ]:
import pandas as pd, os
os.makedirs("/kaggle/working", exist_ok=True) if os.path.exists("/kaggle") else None
OUT = "/kaggle/working" if os.path.exists("/kaggle") else "."

HINT = {"definition":"dictionary","data_source":"dictionary","business_rule":"policy","calculation":"spreadsheet",
        "mapping":"catalog","system_capability":"integration_spec","interpretation":"memo"}   # source-type hints for LLM rows

def route(ent, kind, req_text, why):
    """best-matching internal document (synthetic corpus DOCS) using the typed-prior router"""
    q = vec(f"{req_text} {ent} {why}")
    sc = {d: cos(q, vec(t)) + (0.15 if ty in PRIOR[kind] else 0) for d,(ty,t) in DOCS.items()}
    top = sorted(sc, key=lambda d: -sc[d])[0]
    return f"{top}: {DOCS[top][1].split(':')[0]}"

rows = []
for flags, method, conf in [(a,"slot check","high"), (b,"provenance/deviation","high"), (c,"lexical implicit","review")]:
    for r, ent, kind, why in flags:
        rows.append(dict(requirement_id=r, requirement_text=BRD[r]["text"], gap_type=kind, detail=why,
                         method=method, confidence=conf, suggested_source=route(ent, kind, BRD[r]["text"], why)))
# LLM derivability results (only if cells 11-13 were run)
if "DERIV" in globals():
    for qid, res in DERIV.items():
        for m in res.get("missing", []):
            rows.append(dict(requirement_id=qid, requirement_text=REAL_BRD[qid]["text"], gap_type=m.get("type"),
                             detail=m.get("what"), method="llm derivability", confidence="review",
                             suggested_source="(type hint) " + HINT.get(m.get("type"), "")))
df = pd.DataFrame(rows).sort_values(["requirement_id","confidence"])
df.insert(0, "review_status", "")      # reviewer fills: confirmed / false positive / needs info
df.insert(1, "owner", "")
df.to_csv(f"{OUT}/gap_report.csv", index=False)
try: df.to_excel(f"{OUT}/gap_report.xlsx", index=False)
except Exception as e: print("xlsx skipped:", e)
print(len(df), "rows written to", OUT); df.head(12)
